RETRIEVER

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader,get_response_synthesizer
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Settings

Settings.chunk_size = 128
Settings.chunk_overlap = 50

documents = SimpleDirectoryReader("/mnt/disk1/sjw/llama_index/data").load_data()
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(verbose=True, similarity_top_k=2)
response_synthesizer = get_response_synthesizer(
    response_mode="compact",
)
query_engine = index.as_query_engine()

In [2]:
eval_q_data = SimpleDirectoryReader("/mnt/disk1/sjw/llama_index/eval-q").load_data()

In [3]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [4]:
retriever_result = {}
search_results = []

for count, question in enumerate(questions, start=1):
    search_this = f"{question}"
    search_result = retriever.retrieve(search_this)
    search_results = []
    
    for i in range(retriever.similarity_top_k):
        search_results.append(search_result[i].get_text())

    retriever_result[count] = search_results

print(retriever_result)

{1: ["This radical transformation, made possible by the groundbreaking discovery of a fourth spatial dimension, promises to reshape civilization as we know it. The current understanding of Earth's atmosphere and the three-dimensional space we inhabit has long been a foundation of scientific knowledge. However, recent theoretical advancements have unveiled the possibility of a fourth dimension, a concept that not only challenges our perception of reality but also opens the door to unprecedented technological innovations.", 'In practical terms, fourth-dimensional manipulation might allow for the creation of superfluids with no friction, enhancing the efficiency of oxygen delivery within a liquid atmosphere. Additionally, the discovery of exotic matter with negative mass could enable novel technologies, such as advanced levitation and anti-gravity systems, further transforming human capabilities and infrastructure.\r\n\r\nThe journey from theoretical physics to practical application invol

GENERATOR

In [5]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)

In [6]:
import torch
from transformers import pipeline

pipe = pipeline("text-generation", model="HuggingFaceH4/zephyr-7b-beta", torch_dtype=torch.bfloat16, device_map="auto")


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

We've detected an older driver with an RTX 4000 series GPU. These drivers have issues with P2P. This can affect the multi-gpu inference when using accelerate device_map.Please make sure to update your driver to the latest version which resolves this.


Generator

In [7]:
responses=[]

In [8]:
for count, question in enumerate(questions, start=1):
    retriever_context_list = retriever_result.get(count, [])
    retriever_context = ' '.join(retriever_context_list)
    
    query = f"Consider the following context: '{retriever_context}'. Answer true or false question '{question}'. You have to answer using only True or False without any other explanation. Answer: "
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": query},
    ]
    prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = pipe(prompt, max_new_tokens=256, do_sample=False, temperature=0.0, top_k=50, top_p=0.95)

    generated_text = response[0]["generated_text"]
    
    # Find the position of "Answer:" in the generated text
    aa = generated_text.find("<|assistant|>")
    if aa != -1:
        response_text = generated_text[aa + len("<|assistant|>"):].strip().split()[0]
    else:
        response_text = "Not found"
    
    responses.append(response_text)
    print(count, response_text)

/home/jjh_test/anaconda3/envs/torch2/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jjh_test/anaconda3/envs/torch2/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


1 True.
2 False.
3 True.
4 False.
5 False.
6 False.
7 True.
8 True.
9 True:


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


10 False.
11 False.
12 True.
13 False.
14 True.
15 False.
16 False.
17 False.
18 True.
19 False.
20 True.
21 False.
22 False.
23 False.
24 False.
25 True.
26 False.
27 True.
28 False.
29 True.
30 True.
31 False.
32 True.
33 True.
34 False.
35 True.
36 True.
37 False.
38 False.
39 True.
40 False.
41 True.
42 False.
43 False.
44 True.
45 True.
46 False.
47 True.
48 False.
49 True.
50 False.
51 True.
52 False.
53 False.
54 True.
55 True.
56 True.
57 False:
58 True.
59 False.
60 False.
61 False.
62 True.
63 False.
64 False.
65 False.
66 True.
67 Answer:
68 False.
69 True.
70 False.
71 False.
72 False.
73 True.
74 False.
75 True.
76 False.
77 False.
78 False.
79 True.
80 False.
81 True.
82 False.
83 False.
84 False.
85 True.
86 False.
87 False.
88 False.
89 False.
90 False.
91 False.
92 False.
93 False.
94 False.
95 True.
96 False.
97 True.
98 False.
99 False.
100 False.
101 False.
102 False.
103 False.
104 True.
105 False.
106 True.
107 False.
108 False.
109 False.
110 False.
111 False.
11

EVALUATOR

In [9]:
eval_a_data = SimpleDirectoryReader("/mnt/disk1/sjw/llama_index/eval-a").load_data()

In [10]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [11]:
responses_str = []

In [12]:
for response in responses:
    response_str = str(response)
    if "True" in response_str:
        response_str = "True"
    elif "False" in response_str:
        response_str = "False"
    
    responses_str.append(response_str)

In [13]:
correct_count=0
number=0

for answer, response, response_str, question in zip(answers, responses, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

<<wrong>>
 5 True or False: Scientists plan to use particle colliders to manipulate the fourth dimension.
RESPONSE False.
CORRECT ANSWER True

<<wrong>>
 6 True or False: The discovery of a fourth spatial dimension led to the concept of a liquid atmosphere replacing Earth's air.
RESPONSE False.
CORRECT ANSWER True

<<wrong>>
 9 True or False: Fourth-dimensional technology will not affect the efficiency of oxygen delivery in the liquid atmosphere.
RESPONSE True:
CORRECT ANSWER False

<<wrong>>
 10 True or False: The discovery of exotic matter with negative mass is a direct result of fourth-dimensional research.
RESPONSE False.
CORRECT ANSWER True

<<wrong>>
 16 True or False: The development of nanobots to monitor the liquid atmosphere is a result of overcoming technological challenges.
RESPONSE False.
CORRECT ANSWER True

<<wrong>>
 22 True or False: Improved climate regulation could be a benefit of a liquid atmosphere, driven by fourth-dimensional technology.
RESPONSE False.
CORRECT A

In [14]:
accuracy = (correct_count / len(questions)) * 100
print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")

Total Questions: 200
Correct Answers: 155
Accuracy: 77.50%
